BRONZE LAYER


In [0]:
%sql
USE CATALOG `bronze_catalog`;
USE SCHEMA retail_bronze;

DROP TABLE IF EXISTS retail_bronze.bronze_customers;
DROP TABLE IF EXISTS retail_bronze.bronze_products;
DROP TABLE IF EXISTS retail_bronze.bronze_stores;
DROP TABLE IF EXISTS retail_bronze.bronze_sales;



CREATE TABLES FOR BRONZE LAYER

In [0]:
%sql
--Customer Table

CREATE TABLE retail_bronze.bronze_customers
USING CSV
OPTIONS (
    path = 's3://retail-etl-lakehouse/landing/customers/',
    header = 'true',
    inferSchema = 'true'
);

In [0]:
%sql
--Products Table

CREATE TABLE retail_bronze.bronze_products
USING CSV
OPTIONS (
    path = 's3://retail-etl-lakehouse/landing/products/',
    header = 'true',
    inferSchema = 'true'
);

In [0]:
%sql
--Stores Table

CREATE TABLE retail_bronze.bronze_stores
USING CSV
OPTIONS (
    path = 's3://retail-etl-lakehouse/landing/stores/',
    header = 'true',
    inferSchema = 'true'
);

In [0]:
%sql
--Sales Table

CREATE TABLE retail_bronze.bronze_sales
USING CSV
OPTIONS (
    path = 's3://retail-etl-lakehouse/landing/sales/',
    header = 'true',
    inferSchema = 'true'
);

In [0]:
%sql
select count(*) from retail_bronze.bronze_sales


In [0]:
%sql
-- ============================================================================
-- BRONZE LAYER DATA QUALITY VALIDATIONS
-- ============================================================================

-- Validation 1: Row Count Check - Ensure all tables have data
-- ============================================================================
SELECT 
    'bronze_customers' AS table_name,
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN '✓ PASS' ELSE '✗ FAIL' END AS validation_status
FROM retail_bronze.bronze_customers

UNION ALL

SELECT 
    'bronze_products' AS table_name,
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN '✓ PASS' ELSE '✗ FAIL' END AS validation_status
FROM retail_bronze.bronze_products

UNION ALL

SELECT 
    'bronze_stores' AS table_name,
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN '✓ PASS' ELSE '✗ FAIL' END AS validation_status
FROM retail_bronze.bronze_stores

UNION ALL

SELECT 
    'bronze_sales' AS table_name,
    COUNT(*) AS row_count,
    CASE WHEN COUNT(*) > 0 THEN '✓ PASS' ELSE '✗ FAIL' END AS validation_status
FROM retail_bronze.bronze_sales

ORDER BY table_name;

In [0]:
%sql
-- ============================================================================
-- Validation 2: Null Value Analysis - Check for missing data
-- ============================================================================

-- Customers Table Null Check
SELECT 
    'bronze_customers' AS table_name,
    'CustomerID' AS column_name,
    SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS null_count,
    COUNT(*) AS total_rows
FROM retail_bronze.bronze_customers

UNION ALL

-- Products Table Null Check
SELECT 
    'bronze_products',
    'ProductID',
    SUM(CASE WHEN ProductID IS NULL THEN 1 ELSE 0 END),
    COUNT(*)
FROM retail_bronze.bronze_products

UNION ALL
-- Stores Table Null Check
SELECT 
    'bronze_stores',
    'StoreID',
    SUM(CASE WHEN StoreID IS NULL THEN 1 ELSE 0 END),
    COUNT(*)
FROM retail_bronze.bronze_stores

UNION ALL
-- Sales Table Null Check
SELECT 
    'bronze_sales',
    'TransactionID',
    SUM(CASE WHEN TransactionID IS NULL THEN 1 ELSE 0 END),
    COUNT(*)
FROM retail_bronze.bronze_sales

ORDER BY table_name, column_name;